In [ ]:
## Run the Application in Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ghiridharravi-athenatec/claude_code_flow/blob/main/run_in_colab.ipynb)

# CDI Scorer -- Run in Colab (ngrok tunnel)

Clones the CDI Scorer repo, installs backend + frontend dependencies, starts both servers, and exposes them publicly via [ngrok](https://ngrok.com) so you can use the app from a normal browser tab.

**Before running:** you'll need a free ngrok account and its authtoken (https://dashboard.ngrok.com/get-started/your-authtoken) -- you'll be prompted for it below. You'll also need your own Anthropic (Claude) API key ready, but you paste that into the *app's* UI once it's running, never into this notebook.

Run the cells in order, top to bottom. This notebook assumes a Linux environment with Node.js/npm and git already available (true of the standard Colab runtime).

## 1. Configuration

In [ ]:
# TODO: replace with your repository's URL
REPO_URL = "https://github.com/ghiridharravi-athenatec/claude_code_flow.git"
BRANCH = "main"
REPO_DIR = "cdi-scorer-repo"

BACKEND_PORT = 5000
FRONTEND_PORT = 3000

## 2. Clone the repository

In [ ]:
import os

if "<your-username>" in REPO_URL:
    raise ValueError("Set REPO_URL in the Configuration cell above to your actual repository URL before continuing.")

if os.path.isdir(REPO_DIR):
    print(f"'{REPO_DIR}' already exists -- skipping clone. Delete the folder and re-run this cell to re-clone.")
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}

REPO_PATH = os.path.abspath(REPO_DIR)
BACKEND_DIR = os.path.join(REPO_PATH, "backend")
FRONTEND_DIR = os.path.join(REPO_PATH, "frontend")

print("Repo:    ", REPO_PATH)
print("Backend: ", BACKEND_DIR)
print("Frontend:", FRONTEND_DIR)

## 3. Install backend dependencies

In [ ]:
# libmagic is the native library python-magic wraps (SPEC.md Section 1.1) --
# Colab's Debian base doesn't ship it by default.
!apt-get -qq update && apt-get install -y -qq libmagic1

!pip install -q -r {BACKEND_DIR}/requirements.txt

print("Backend dependencies installed.")

Optional: pre-download the spaCy model used for PHI redaction on the backend's unhandled-error logging path (~400MB). Not required for the normal upload -> score flow -- skip unless you specifically want it warmed up in advance.

In [ ]:
# !python -m spacy download en_core_web_lg

## 4. Install frontend dependencies

In [ ]:
!npm install --prefix {FRONTEND_DIR}

## 5. Install and configure pyngrok

In [ ]:
!pip install -q pyngrok requests

import getpass
from pyngrok import ngrok

ngrok_authtoken = getpass.getpass("Enter your ngrok authtoken: ")
ngrok.set_auth_token(ngrok_authtoken)
del ngrok_authtoken  # never keep the token around longer than needed

print("ngrok configured.")

## 6. Open the tunnels

Both tunnels are opened *before* either server starts. The backend needs the frontend's public URL for CORS, and the frontend needs the backend's public URL to call the API -- opening both first (ngrok allocates a public URL immediately, even before anything is listening locally) lets each server start already knowing the other's address.

In [ ]:
from pyngrok import ngrok
from pyngrok.exception import PyngrokNgrokError


def open_tunnel(port):
    for existing in ngrok.get_tunnels():
        if existing.config["addr"].endswith(f":{port}"):
            return existing
    return ngrok.connect(port, "http")


try:
    backend_tunnel = open_tunnel(BACKEND_PORT)
    frontend_tunnel = open_tunnel(FRONTEND_PORT)
except PyngrokNgrokError as exc:
    raise RuntimeError(
        "Could not open both ngrok tunnels. Free ngrok accounts are sometimes "
        "limited to one simultaneous tunnel -- if so, you'll need a paid plan "
        "or a second authtoken/session to run frontend + backend together."
    ) from exc

backend_public_url = backend_tunnel.public_url
frontend_public_url = frontend_tunnel.public_url

print("Backend tunnel: ", backend_public_url)
print("Frontend tunnel:", frontend_public_url)

## 7. Start the backend

In [ ]:
import os
import subprocess
import sys
import time

import requests


def wait_until_up(url, timeout=90):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            if requests.get(url, timeout=2).status_code == 200:
                return True
        except requests.exceptions.RequestException:
            pass
        time.sleep(1)
    return False


backend_log_path = os.path.join(REPO_PATH, "backend_server.log")
backend_log = open(backend_log_path, "w")

backend_env = os.environ.copy()
backend_env["ALLOWED_ORIGIN"] = frontend_public_url

if "backend_process" in globals() and backend_process.poll() is None:
    print("Backend already running (PID", backend_process.pid, ")")
else:
    backend_process = subprocess.Popen(
        [sys.executable, "app.py"],
        cwd=BACKEND_DIR,
        env=backend_env,
        stdout=backend_log,
        stderr=subprocess.STDOUT,
    )
    print("Started backend (PID", backend_process.pid, ") -- logging to", backend_log_path)

if wait_until_up(f"http://localhost:{BACKEND_PORT}/api/v1/health"):
    print("Backend is up.")
else:
    print(f"Backend did not become healthy in time -- check {backend_log_path}")

## 8. Build and serve the frontend

A production build is served with a plain static file server rather than the React dev server -- the dev server rejects requests arriving through an ngrok tunnel with an "Invalid Host header" error by default.

In [ ]:
frontend_build_env = os.environ.copy()
frontend_build_env["REACT_APP_API_BASE_URL"] = f"{backend_public_url}/api/v1"

print("Building frontend (this can take a minute)...")
subprocess.run(
    ["npm", "run", "build"],
    cwd=FRONTEND_DIR,
    env=frontend_build_env,
    check=True,
)
print("Frontend build complete.")

In [ ]:
frontend_log_path = os.path.join(REPO_PATH, "frontend_server.log")
frontend_log = open(frontend_log_path, "w")

if "frontend_process" in globals() and frontend_process.poll() is None:
    print("Frontend already running (PID", frontend_process.pid, ")")
else:
    frontend_process = subprocess.Popen(
        [
            sys.executable,
            "-m",
            "http.server",
            str(FRONTEND_PORT),
            "--directory",
            os.path.join(FRONTEND_DIR, "build"),
        ],
        cwd=FRONTEND_DIR,
        stdout=frontend_log,
        stderr=subprocess.STDOUT,
    )
    print("Started frontend (PID", frontend_process.pid, ") -- logging to", frontend_log_path)

if wait_until_up(f"http://localhost:{FRONTEND_PORT}"):
    print("Frontend is up.")
else:
    print(f"Frontend did not become reachable in time -- check {frontend_log_path}")

## 9. Published URLs

In [ ]:
print("Open the app here:")
print(f"  App (frontend): {frontend_public_url}")
print(f"  API (backend):  {backend_public_url}/api/v1")
print()
print("The app will ask for your Claude API key in the browser -- it is never")
print("entered here, and is used only client-side plus for the single request")
print("it authorizes (SPEC.md Section 6).")

## 10. Shut down (optional)

Run this when you're done to stop both servers and close the tunnels.

In [ ]:
for proc_name in ("backend_process", "frontend_process"):
    proc = globals().get(proc_name)
    if proc is not None and proc.poll() is None:
        proc.terminate()
        print(f"Stopped {proc_name} (PID {proc.pid})")

ngrok.kill()
print("ngrok tunnels closed.")